In [ ]:
import os
import csv
from pathlib import Path

import torch
from PIL import Image
from transformers import pipeline
from huggingface_hub import login

# -----------------------------
# AUTHENTICATION (ADD TOKEN HERE)
# -----------------------------
HF_TOKEN = os.environ["HF_TOKEN"]     # <--- put your actual token
login(HF_TOKEN)


# -----------------------------
# CONFIG – EDIT THESE
# -----------------------------
INPUT_DIR = "./data/LLaVA-Med/images/ADR_images"
OUTPUT_CSV = "./data/LLaVA-Med/MedGemma_Localization_D1500_07Dec2025_V1.csv"

MAX_NEW_TOKENS = 256
# -----------------------------


def get_device_and_dtype():
    """Choose device and dtype based on GPU availability."""
    if torch.cuda.is_available():
        device = "cuda"
        dtype = torch.bfloat16
    else:
        device = "cpu"
        dtype = torch.float32
    return device, dtype


def init_medgemma_pipeline():
    """Initialize MedGemma 4B image-text-to-text pipeline."""
    device, dtype = get_device_and_dtype()
    print(f"Using device: {device}, dtype: {dtype}")

    pipe = pipeline(
        task="image-text-to-text",
        model="google/medgemma-4b-it",
        torch_dtype=dtype,
        device=device,
    )
    return pipe


def iter_image_files(folder: Path):
    """Yield image paths from folder (non-recursive)."""
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    for p in sorted(folder.iterdir()):
        if p.is_file() and p.suffix.lower() in exts:
            yield p


def build_messages(prompt: str, image: Image.Image):
    """Build chat-style messages for MedGemma 4B."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    return messages

pipe = init_medgemma_pipeline()


In [ ]:
# -----------------------------
# CONFIG – EDIT THESE
# -----------------------------
INPUT_DIR = "./data/LLaVA-Med/images/multimodal adr"
OUTPUT_CSV = "./data/LLaVA-Med/MedGemma_Localization_D1500_07Dec2025_V4.csv"

PROMPT = """You are a clinical diagnostic assistant.
Analyze the provided image for visual evidence of an Adverse Drug Effect (ADE).

If an ADE is detected, output only the location of the ADE (no explanation, no extra words).

If an ADE is not detected, still output only the body part visible in the same format.

The output must be just a short phrase, like the examples below.

Output examples (few-shot demonstrations):

Example 1:
Image: (patient with oral ulcers visible)
Response: tongue

Example 2:
Image: (patient with swollen left leg due to drug-induced edema)
Response: left leg

Example 3:
Image: (image shows abdominal rash consistent with ADE)
Response: stomach

Example 4:
Image: (no ADE visible, only arm shown)
Response: right arm

Example 5:
Image: (no ADE visible, only face shown)
Response: face"""

import csv
import re
from pathlib import Path
from PIL import Image

# helper to remove control chars and collapse whitespace/newlines
def sanitize_for_csv(s: str, max_len: int = None) -> str:
    if s is None:
        return ""
    # remove null bytes and other weird control characters except typical whitespace
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]+", " ", s)
    # Convert all types of newlines to a single space
    s = s.replace("\r", " ").replace("\n", " ")
    # collapse multiple spaces
    s = re.sub(r"\s{2,}", " ", s).strip()
    if max_len is not None and len(s) > max_len:
        return s[: max_len - 3] + "..."
    return s

def run_inference_on_folder(
    pipe,
    input_dir: str,
    output_csv: str,
    prompt: str,
    max_new_tokens: int = 256,
):
    input_path = Path(input_dir)
    if not input_path.exists():
        raise FileNotFoundError(f"Input folder not found: {input_path.resolve()}")

    image_paths = list(iter_image_files(input_path))
    if not image_paths:
        raise RuntimeError(f"No image files found in: {input_path.resolve()}")

    print(f"Found {len(image_paths)} image(s) in {input_path.resolve()}")

    # Use utf-8-sig so Excel on Windows recognizes UTF-8, and force quoting.
    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["filename", "generated_text", "prompt"])

        # Write prompt ONCE (sanitized)
        writer.writerow(["", "", sanitize_for_csv(prompt)])

        #i = 0
        #top = 10
        for img_path in image_paths:
            #i += 1
            #if i > top:
                #break

            print(f"\nProcessing: {img_path.name}...")

            try:
                image = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"  [ERROR] Failed to open image {img_path}: {e}")
                continue

            messages = build_messages(prompt, image)

            try:
                output = pipe(
                    text=messages,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
                # Extract text (keep as str)
                generated = output[0]["generated_text"][-1]["content"]
                print(f"  → OUTPUT: {generated}")

            except Exception as e:
                print(f"  [ERROR] Inference failed for {img_path.name}: {e}")
                generated = f"[ERROR] {e}"

            # Sanitize before writing (remove newlines/control chars)
            clean = sanitize_for_csv(generated, max_len=10000)  # adjust max_len if you want
            writer.writerow([img_path.name, clean, ""])

    print(f"\nDone! Results saved to: {Path(output_csv).resolve()}")



#if __name__ == "__main__":

run_inference_on_folder(
    pipe=pipe,
    input_dir=INPUT_DIR,
    output_csv=OUTPUT_CSV,
    prompt=PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
)
